# 00 · Setup and global settings

This notebook holds everything that the other notebooks share: the environment, the **global settings** (the master folder and every path below it),
the constants of the study, and the functions that more than one notebook needs (signal processing, the three foundation models, access to the cached recordings).

Every other notebook starts with

```python
%run ./00_setup_and_settings.ipynb
```

so the settings are defined in one place only. To use the code on your computer, edit the cell **Global settings** below (or set the environment variable `ECGREP_MASTER`).

## Order of the notebooks

| Notebook | What it does | Needs |
|---|---|---|
| `00_setup_and_settings` | environment, global settings, shared functions | – |
| `01_data_download_and_preprocessing` | where to obtain the four datasets, download, filtering, cached arrays | internet, about 150 GB of disk space |
| `02_training_and_evaluation_sets` | exclusions, waveform check, training pool, evaluation sets, pairs of recordings | output of 01 |
| `03_model_adaptation` | pretrained weights; fine-tuning of the baseline, the teacher and the student, and of the comparison models | GPU, output of 02 |
| `04_reading_and_penalized_head` | single read, window readings, penalized head, Monte Carlo dropout; writes the outputs of every model for every evaluation recording | GPU, output of 03 |
| `05_analysis` | decision thresholds, decision change, AUROC, direction agreement, paired bootstrap | output of 04 |
| `06_computational_cost` | time of adaptation and of reading | GPU |
| `07_tables` | Tables 1–2 and Supplementary Tables 1–4 | output of 02, 05 |
| `08_figures` | Figs. 1–5 and Supplementary Figs. 1–3 | output of 02, 04, 05, 06 |

Data, pretrained weights and fine-tuned models are **not** part of the repository. Notebook 01 explains where each dataset comes from and what access it requires;
notebook 03 explains where the pretrained weights come from. Fine-tuned models are written by notebook 03 and are read from disk by the later notebooks.

## Environment

The analyses were run with Python 3.10 on Windows 11 with one NVIDIA RTX 3080 graphics card (10 GB), CUDA 12.4.
The file `environment.yml` in the root of the repository lists the versions that were used. To create the environment:

```bash
conda env create -f environment.yml
conda activate ecg-fm-repeatability
python -m ipykernel install --user --name ecg-fm-repeatability
jupyter lab
```

or, with pip only (Python 3.10):

```bash
pip install torch==2.6.0 --index-url https://download.pytorch.org/whl/cu124
pip install -r requirements.txt
```

The cell below prints the versions that are installed, so that they can be compared with `environment.yml`.

In [ ]:
import importlib
import platform

print("python", platform.python_version())
for _name in ("torch", "numpy", "pandas", "scipy", "sklearn", "transformers", "safetensors", "neurokit2", "wfdb", "h5py", "matplotlib"):
    try:
        print(f"{_name:13s}", importlib.import_module(_name).__version__)
    except Exception as _e:                                    # a missing package is reported, not raised: notebooks that do not need it still run
        print(f"{_name:13s} not available ({type(_e).__name__})")

## Global settings

`MASTER` is the folder that holds the data, the models and the results. It can be anywhere, also on another drive than the repository.
Nothing is written outside `MASTER`.

```
MASTER/
├── data/raw/            downloaded datasets, never modified          (notebook 01)
│   ├── ptb-xl/          ptbxl_database.csv, scp_statements.csv, records500/
│   ├── mimic-iv-ecg/    record_list.csv, machine_measurements.csv, files/
│   ├── mimic-iv/hosp/   patients.csv.gz   (optional: age and sex for Table 1)
│   ├── code15/          exams.csv, exams_part0.zip … exams_part17.zip
│   └── echonext/        echonext_metadata_100k.csv, EchoNext_*_waveforms.npy
├── data/derived/        filtered recordings and the tables of the sets (notebooks 01, 02)
├── models/pretrained/   ECGFounder, HuBERT-ECG, ECG-FM as released   (notebook 03)
├── models/finetuned/    one file per fine-tuned model                (notebook 03)
├── results/             teacher targets, model outputs, summaries    (notebooks 03–06)
└── output/              tables and figures of the paper              (notebooks 07, 08)
```

In [ ]:
# ============================== GLOBAL SETTINGS ==============================
import os
from pathlib import Path

# --- the master folder: edit this line, or set the environment variable ECGREP_MASTER --------------------------------
MASTER = Path(os.environ.get("ECGREP_MASTER", "D:/ecg_fm_repeatability"))

# --- raw data (downloaded in notebook 01) ------------------------------------------------------------------------------
RAW = MASTER / "data" / "raw"
PTBXL_DIR = RAW / "ptb-xl"                                   # PTB-XL 1.0.3: ptbxl_database.csv, scp_statements.csv, records500/
MIMIC_DIR = RAW / "mimic-iv-ecg"                             # MIMIC-IV-ECG 1.0: record_list.csv, machine_measurements.csv, files/
MIMIC_PATIENTS = RAW / "mimic-iv" / "hosp" / "patients.csv.gz"   # MIMIC-IV (credentialed access); optional, only for age and sex in Table 1
CODE15_DIR = RAW / "code15"                                  # CODE-15%: exams.csv, exams_part{0..17}.zip (or the extracted .hdf5 files)
ECHONEXT_DIR = RAW / "echonext"                              # EchoNext 1.1.1: echonext_metadata_100k.csv, EchoNext_{train,val,test,no_split}_waveforms.npy

# --- derived data (written by notebooks 01 and 02) ------------------------------------------------------------------------
DERIVED = MASTER / "data" / "derived"
PTBXL_ARRAY = DERIVED / "ptbxl_filtered_f16.npy"             # (n, 12, 5000) float16, mV, 500 Hz; rows in the order of ptbxl_database.csv
CODE15_CACHE = DERIVED / "code15"                            # part{i}_filtered_f16.npy and part{i}_index.csv for i = 0..17
ECHONEXT_ARRAY = DERIVED / "echonext" / "filtered500_f16.npy"    # (100000, 12, 5000) float16
ECHONEXT_INDEX = DERIVED / "echonext" / "index.csv"          # row of the array for every row of the metadata
MIMIC_CHECK = DERIVED / "mimic_waveform_check.csv"           # waveform check of the MIMIC-IV-ECG recordings that were examined
MIMIC_LABELS = DERIVED / "mimic_machine_labels.csv"          # labels derived from the machine statements, all recordings
MIMIC_POOL_ARRAY = DERIVED / "mimic_pool_f16.npy"            # recordings of the unlabelled training pool
MIMIC_EVAL_ARRAY = DERIVED / "mimic_eval_f16.npy"            # recordings of the MIMIC-IV-ECG evaluation sets
SETS = DERIVED / "sets"
PTBXL_SETS = SETS / "ptbxl_sets.csv"                         # one row per PTB-XL recording: fold, set, labels
POOL_RECORDS = SETS / "mimic_pool_records.csv"               # one row per recording of the unlabelled training pool
EVAL_RECORDS = SETS / "eval_records.csv"                     # one row per evaluation recording (all datasets); the order of the rows is the order of all model outputs
SET_COUNTS = SETS / "set_counts.json"                        # numbers examined, excluded and selected (flow diagram, Table 1)

# --- models -------------------------------------------------------------------------------------------------------------
PRETRAINED = MASTER / "models" / "pretrained"
ECGFOUNDER_CODE = PRETRAINED / "ECGFounder"                  # clone of https://github.com/PKUDigitalHealth/ECGFounder (net1d.py)
ECGFOUNDER_WEIGHTS = PRETRAINED / "12_lead_ECGFounder.pth"
HUBERT_DIR = PRETRAINED / "hubert_ecg"                       # model.safetensors and config.json of Edoardo-Coppola/hubert-ecg-base
ECGFM_WEIGHTS = PRETRAINED / "ecg_fm" / "mimic_iv_ecg_physionet_pretrained.pt"   # wanglab/ecg-fm
FINETUNED = MASTER / "models" / "finetuned"                  # {tag}.pt, see model_tag() below

# --- results --------------------------------------------------------------------------------------------------------------
RESULTS = MASTER / "results"
TRAINING_LOGS = RESULTS / "training_logs"                    # validation AUROC during training
TEACHER_TARGETS = RESULTS / "teacher_targets"                # outputs of each teacher for the training segments
LOGITS = RESULTS / "logits"                                  # {tag}.npz: outputs of a fine-tuned model for every row of EVAL_RECORDS
MC_DROPOUT = RESULTS / "mc_dropout"                          # {tag}.npz: Monte Carlo dropout probabilities (PTB-XL)
SUMMARY = RESULTS / "summary"                                # outcome tables with bootstrap confidence intervals
TIMING = RESULTS / "timing"                                  # measured times
FIGURES = MASTER / "output" / "figures"
TABLES = MASTER / "output" / "tables"

for _d in (RAW, DERIVED, SETS, CODE15_CACHE, ECHONEXT_ARRAY.parent, PRETRAINED, FINETUNED, TRAINING_LOGS, TEACHER_TARGETS, LOGITS, MC_DROPOUT, SUMMARY, TIMING, FIGURES, TABLES):
    _d.mkdir(parents=True, exist_ok=True)
print("MASTER =", MASTER)

## Constants of the study

These values define the design that is described in the Methods of the paper. They are collected here so that no notebook repeats them.

In [ ]:
# --- signal ---------------------------------------------------------------------------------------------------------------
FS = 500                      # sampling rate of every cached recording (Hz)
N10 = 10 * FS                 # samples in a 10-s recording
SEG = 5 * FS                  # samples in a 5-s segment: a model reads one segment as its ECG
WIN = int(2.5 * FS)           # samples in a 2.5-s window
VIEW = int(4.5 * FS)          # samples in a 4.5-s view (penalized head, agreement loss)
VIEW_SHIFT = int(0.5 * FS)    # the two views of the penalized head start 0.5 s apart
N_WIN_READ = 25               # windows of 2.5 s in a window reading (evenly spaced onsets, first at the start, last ending at the end of the segment)
N_WIN_TEACHER = 7             # windows of 2.5 s whose teacher outputs are averaged to give the distillation target

# --- diagnostic classes ---------------------------------------------------------------------------------------------------
CLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]
CLASS_NAMES = {"NORM": "Normal ECG", "MI": "Myocardial infarction", "STTC": "ST/T change", "CD": "Conduction disturbance", "HYP": "Hypertrophy"}

# --- foundation models and fine-tuned models ------------------------------------------------------------------------------
MODELS = ["ecgfounder", "hubert", "ecgfm"]
MODEL_NAMES = {"ecgfounder": "ECGFounder", "hubert": "HuBERT-ECG", "ecgfm": "ECG-FM"}
SEEDS = [0, 1, 2]                                   # every configuration was trained with three random seeds
ENSEMBLE_SEEDS = list(range(10))                    # ECGFounder baseline models averaged in the ensemble
DROPOUT_RATES = [0.1, 0.2, 0.3, 0.4, 0.5]           # ECGFounder baseline trained with dropout (Monte Carlo dropout); the rate with the highest validation AUROC is reported
# kinds of fine-tuned model (the first three exist for all foundation models, the others for ECGFounder only)
#   baseline     labels only, the whole 5-s segment as input
#   teacher      labels only, random 2.5-s windows as input (source of the distillation target)
#   student      distillation from the teacher (the training remedy)
#   windows      as the teacher, but with all layers fine-tuned (alternative: training on 2.5-s windows)
#   agreement    direct agreement loss between two shifted views (alternative)
#   selfdistill  distillation of the window average of the baseline model itself (alternative)
#   dropout10 … dropout50   baseline trained with dropout at a rate of 0.1 … 0.5


def model_tag(model, kind, seed):
    """Name of a fine-tuned model, for example model_tag("ecgfounder", "student", 0) -> "ecgfounder_student_s0"."""
    return f"{model}_{kind}_s{seed}"


# --- the three remedies ----------------------------------------------------------------------------------------------------
MIX_SINGLE = 0.1              # mixed reading: 0.1 x probability of one read + 0.9 x mean probability over the 25 windows
PENALTY_WEIGHT = 300          # penalty weight (gamma) of the penalized head in the final setting
PENALTY_WEIGHTS = [0, 30, 100, 300, 1000]           # weights that were fitted; 0 = head refitted without a penalty
HEAD_C = 0.01                 # inverse strength of the ridge penalty of the refitted logistic regression

# --- evaluation -----------------------------------------------------------------------------------------------------------
SENSITIVITY = 0.90            # decision thresholds: 90% sensitivity in the PTB-XL validation fold, for each class and each configuration
N_BOOT = 2000                 # bootstrap resamples for decision change and direction agreement
N_BOOT_AUROC = 1000           # bootstrap resamples for the AUROC
BOOT_SEED = 20261117          # seed of the bootstrap; every evaluation set has its own generator
# names of the evaluation sets (column "set" of EVAL_RECORDS)
EVAL_SETS = ["ptbxl_val", "ptbxl_test", "mimic_test", "mimic_pairs_all", "mimic_pairs_same_report", "code15_test", "code15_cd_change", "code15_normal_change", "echonext_test"]

## Signal processing

Functions that notebooks 01 and 02 share: the filter that every recording passes before it is cached, the waveform check,
the access to single recordings of MIMIC-IV-ECG, and a helper that runs a function on many recordings in parallel.

### Filter

Every recording is filtered once, as a whole 10-s recording at 500 Hz, before any segment or window is cut from it.
The filter follows the preprocessing of the ECGFounder repository (`filter_bandpass` in its `util.py`), and the same filtered recordings are the input of all three foundation models:

1. a notch filter at 50 Hz (quality factor 30) against power-line interference;
2. a Butterworth band-pass filter of order 4 with a pass band of 0.67–40 Hz;
3. removal of the baseline, which is estimated by a running median over 0.4 s plus one sample (201 samples at 500 Hz) and subtracted.

Steps 1 and 2 are applied forwards and backwards, so that they do not shift the signal in time.
`filter_ecg` is our own implementation with SciPy. It was compared with the function of the ECGFounder repository on recordings of all four datasets and returned identical values
(largest absolute difference 0; the last section of notebook 01 repeats the comparison).

In [ ]:
import hashlib
import threading
import time

import numpy as np
from scipy.signal import butter, filtfilt, iirnotch, medfilt

LEADS = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]   # order of the leads in every cached array; lead II is index 1
MIN_R_WAVES = 3                                               # waveform check: at least this many R waves in each 5-s half of lead II
CODE15_MIN_SAMPLES = 3868                                     # CODE-15%: exams with at least this many samples of signal (9.67 s at 400 Hz) in the 10-s window count as complete 10-s exams
N_JOBS = max(1, min(16, (os.cpu_count() or 2) - 2))           # worker processes for filtering and for the waveform check; set to 1 to run everything in the notebook process
SET_CHARACTERISTICS = SETS / "set_characteristics.csv"        # recordings, patients, age, sex, labels and pair intervals of every set (written by notebook 02; Table 1, Supplementary Table 1)


def filter_ecg(x, fs=FS):
    """Filters one recording. x: array (leads, samples), sampled at fs Hz. Returns a float64 array of the same shape.
    Same operations, in the same order and with the same parameters, as the preprocessing of the ECGFounder repository."""
    x = np.asarray(x, dtype=np.float64)
    b, a = iirnotch(50.0, 30.0, fs)                                        # 1. power-line interference
    y = filtfilt(b, a, x, axis=-1)
    b, a = butter(4, [0.67, 40.0], btype="bandpass", fs=fs)                # 2. band-pass
    y = filtfilt(b, a, y, axis=-1)
    width = int(0.4 * fs) + 1                                              # 3. running median over 0.4 s + 1 sample ...
    width += 1 - width % 2                                                 #    ... made odd, so that the window is centred on the sample
    baseline = np.stack([medfilt(lead, kernel_size=width) for lead in y])  #    (medfilt pads the ends of the recording with zeros)
    return y - baseline

### Waveform check

A recording passes the waveform check when at least three R waves are detected in each 5-s half of the filtered lead II.
The reason is that a model reads one half as its ECG: a half with almost no beats cannot be read, whatever the model.
R waves are detected in the whole 10-s recording with NeuroKit2 (`ecg_peaks`, default method) and are then assigned to the half in which they lie.
When the detector raises an error (it does so on some flat or very noisy signals), the recording counts as having no R wave.

In [ ]:
def count_r_waves(lead_ii):
    """Numbers of R waves in the first and in the second 5-s half of a filtered lead II (10 s at 500 Hz)."""
    import neurokit2 as nk
    try:
        _, info = nk.ecg_peaks(np.asarray(lead_ii, dtype=np.float64), sampling_rate=FS)
        r = np.asarray(info["ECG_R_Peaks"], dtype=float)
    except Exception:
        return 0, 0
    return int((r < SEG).sum()), int((r >= SEG).sum())


def passes_waveform_check(n_first, n_second):
    """True when both halves have at least MIN_R_WAVES R waves."""
    return n_first >= MIN_R_WAVES and n_second >= MIN_R_WAVES

### Parallel execution and writing of arrays

Reading and filtering take 5–30 ms per recording, and several hundred thousand recordings are processed. `parallel_map` distributes this work over `N_JOBS` worker processes.
It uses loky, the process pool that is distributed with joblib (a dependency of scikit-learn), because loky sends the function itself to the workers:
functions that are defined in a notebook can then be used on Windows and on Linux alike, which the process pool of the standard library does not allow.

`write_array` stores filtered recordings in the format of all cached arrays: a `.npy` file of shape `(n, 12, 5000)` in float16 (12 leads in the order of `LEADS`, 10 s at 500 Hz).
Float16 halves the size of the files compared with float32; its resolution (1 µV or better for amplitudes below 2 mV) is sufficient for the models, which were fine-tuned and evaluated on these arrays.

In [ ]:
def parallel_map(function, items, n_jobs=None):
    """Yields function(item) for every item, in the order of the items. The calls run in worker processes; with n_jobs = 1 they run in this process.
    At most 8 items per worker are in work or waiting to be used, so results cannot pile up in memory when they are produced faster than they are consumed."""
    n_jobs = N_JOBS if n_jobs is None else n_jobs
    if n_jobs == 1:
        for item in items:
            yield function(item)
        return
    from collections import deque
    from joblib.externals.loky import get_reusable_executor
    executor = get_reusable_executor(max_workers=n_jobs); waiting = deque()
    for item in items:
        waiting.append(executor.submit(function, item))
        if len(waiting) >= 8 * n_jobs:
            yield waiting.popleft().result()
    while waiting:
        yield waiting.popleft().result()


def write_array(path, n_rows, blocks, report_every=0):
    """Writes filtered recordings to a .npy file of shape (n_rows, 12, 5000), float16. `blocks` yields arrays (k, 12, 5000) or single recordings (12, 5000), in the order of the rows.
    The file gets its final name only after the last row has been written, so an interrupted run never leaves an incomplete array under that name."""
    path = Path(path); partial = path.with_name(path.name + ".partial"); done, t0, last = 0, time.time(), 0
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(partial, "wb") as f:
        np.lib.format.write_array_header_1_0(f, {"descr": "<f2", "fortran_order": False, "shape": (int(n_rows), 12, N10)})
        for block in blocks:
            block = np.asarray(block, dtype="<f2").reshape(-1, 12, N10)
            f.write(block.tobytes()); done += len(block)
            if report_every and done - last >= report_every:
                last = done; print(f"   {path.name}: {done:,} of {n_rows:,} recordings, {(time.time() - t0) / 60:.1f} min", flush=True)
    if done != n_rows:
        raise RuntimeError(f"{path.name}: {done} recordings were written, {n_rows} were expected")
    os.replace(partial, path)

### Recordings of MIMIC-IV-ECG

MIMIC-IV-ECG holds 800,035 recordings as separate WFDB records (a header file and a waveform file of 120,000 bytes each), and only the recordings that enter a set are filtered and cached.
The functions below download single records, read them in the lead order of `LEADS` (the files store aVF before aVL), and apply the waveform check.
A record is identified by the column `path` of `record_list.csv`, for example `files/p1000/p10000032/s40689238/40689238`.

A recording cannot be used when its file cannot be read, when it does not have 12 leads of 5,000 samples, or when the signal is zero everywhere (the dataset contains some empty recordings).
The MD5 checksum of the waveform file is stored with the result of the check, because some pairs of recordings of one patient have identical waveform files, and such pairs are not used (notebook 02).

In [ ]:
MIMIC_URL = "https://physionet.org/files/mimic-iv-ecg/1.0/"
MIMIC_DAT_BYTES = 12 * N10 * 2                     # a waveform file: 12 leads x 5,000 samples x 16 bits
_THREAD = threading.local()


def mimic_record_present(path):
    """True when the header and the complete waveform file of a record are in MIMIC_DIR."""
    hea, dat = MIMIC_DIR / f"{path}.hea", MIMIC_DIR / f"{path}.dat"
    return hea.exists() and hea.stat().st_size > 0 and dat.exists() and dat.stat().st_size == MIMIC_DAT_BYTES


def _fetch_mimic_record(path):
    """Downloads the two files of one record unless they exist. Returns "" or a description of the error."""
    import requests
    if getattr(_THREAD, "session", None) is None:
        _THREAD.session = requests.Session()                               # one connection per thread, reused for all its files
    for ext in ("hea", "dat"):
        dst = MIMIC_DIR / f"{path}.{ext}"
        if dst.exists() and (dst.stat().st_size == MIMIC_DAT_BYTES if ext == "dat" else dst.stat().st_size > 0):
            continue
        dst.parent.mkdir(parents=True, exist_ok=True); error = ""
        for attempt in range(6):
            try:
                r = _THREAD.session.get(MIMIC_URL + f"{path}.{ext}", timeout=60)
                if r.status_code == 200 and (len(r.content) == MIMIC_DAT_BYTES if ext == "dat" else len(r.content) > 0):
                    part = dst.with_suffix(dst.suffix + ".part"); part.write_bytes(r.content); os.replace(part, dst); error = ""   # written under another name first: an interrupted download leaves no incomplete file
                    break
                error = f"{ext}: status {r.status_code}, {len(r.content)} bytes"
                if r.status_code == 404:
                    break
            except Exception as e:
                error = f"{ext}: {type(e).__name__}"
            time.sleep(min(60, 2 ** (attempt + 1)))                        # wait longer after every failed attempt
        if error:
            return error
    return ""


def download_mimic_records(paths, connections=16, verbose=True):
    """Downloads the records in `paths` that are not yet in MIMIC_DIR (open access, no account needed). Existing files are kept, so the call can be repeated.
    Raises an error when a record cannot be obtained, so that a failed download is never mistaken for an unreadable recording. Returns the number of records that were downloaded."""
    from concurrent.futures import ThreadPoolExecutor
    todo = [p for p in dict.fromkeys(paths) if not mimic_record_present(p)]
    if not todo:
        return 0
    t0 = time.time(); failed = []
    with ThreadPoolExecutor(connections) as pool:
        for k, (p, error) in enumerate(zip(todo, pool.map(_fetch_mimic_record, todo)), 1):
            if error:
                failed.append((p, error))
            if verbose and (k % 5000 == 0 or k == len(todo)):
                el = time.time() - t0
                print(f"   downloaded {k:,} of {len(todo):,} records | {k / el:.1f} records/s | failed {len(failed):,} | {el / 60:.0f} min", flush=True)
    if failed:
        raise RuntimeError(f"{len(failed)} records could not be downloaded (first: {failed[0]}). Run the cell again; files that are complete are not downloaded again.")
    return len(todo)


def read_mimic_record(path):
    """The 12 leads of one MIMIC-IV-ECG record as a float64 array (12, 5000) in mV at 500 Hz, leads in the order of LEADS; None when the recording cannot be used."""
    import wfdb
    if not mimic_record_present(path):
        raise FileNotFoundError(f"{path} is not in {MIMIC_DIR}: download it first (notebook 01, or download_mimic_records)")
    try:
        signal, header = wfdb.rdsamp(str(MIMIC_DIR / path))
        names = [n.strip().upper() for n in header["sig_name"]]
        x = np.nan_to_num(signal.T[[names.index(lead.upper()) for lead in LEADS]].astype(np.float64))
    except Exception:
        return None
    if x.shape != (12, N10) or np.abs(x).max() == 0:
        return None
    return x


def check_mimic_record(path):
    """Waveform check of one MIMIC-IV-ECG record: (readable 0/1, R waves in the first half, R waves in the second half, MD5 of the waveform file)."""
    x = read_mimic_record(path)
    if x is None:
        return 0, 0, 0, ""
    n_first, n_second = count_r_waves(filter_ecg(x)[1])
    return 1, n_first, n_second, hashlib.md5((MIMIC_DIR / f"{path}.dat").read_bytes()).hexdigest()


def filter_mimic_record(path):
    """The filtered recording of one MIMIC-IV-ECG record as float16 (12, 5000), as it is stored in the cached arrays."""
    x = read_mimic_record(path)
    if x is None:
        raise ValueError(f"{path} cannot be used")
    return filter_ecg(x).astype(np.float16)

## Foundation models

The three foundation models are used through one class, `FoundationModel`, so that training, reading and timing are written once for all of them.

**Standard input.** Every function below takes ECG signals as a float tensor of shape `(batch, 12, samples)`: the 12 leads in the order I, II, III, aVR, aVL, aVF, V1–V6,
at 500 Hz, in mV, cut from the cached filtered recordings. An input has 2,500 samples for a 5-s segment, 1,250 for a 2.5-s window and 2,250 for a 4.5-s view.
Everything that is specific to a model happens inside the class:

| Model | Scaling of the input | What the network receives | Embedding |
|---|---|---|---|
| ECGFounder | mean 0 and SD 1 over all leads and time | 12 channels at 500 Hz | mean over time of the output of the last stage, 1,024 dimensions |
| HuBERT-ECG (base size) | each lead to the range −1 to 1 | 100 Hz, the 12 leads one after another in one sequence, each lead in its own position of 5 s | mean over the tokens of the last transformer layer, 768 dimensions |
| ECG-FM | each lead to mean 0 and SD 1 | 12 channels at 500 Hz | mean over the tokens of the last transformer layer, 768 dimensions |

The scaling uses only the samples of the input itself (a segment, a window or a view) and never the rest of the recording,
because a model that is given a 5-s segment as its ECG has nothing else. For the same reason the original preprocessing of HuBERT-ECG and ECG-FM,
which scales each lead over the whole recording, is applied here within the input.

**Inputs shorter than 5 s.** ECGFounder and ECG-FM take the leads as input channels, so a shorter input is simply a shorter sequence.
HuBERT-ECG takes the leads concatenated in time and recognises a lead by its position in a 5-s layout.
A shorter input is therefore repeated within the position of each lead until the 5 s are filled; without this, the leads of a 2.5-s window would fall in the wrong positions.

**Parts of the class.** `FoundationModel(name)` has the pretrained network in `bb` and a new linear layer with five outputs, one for each diagnostic class, in `head`.
`embed(x)` returns the embedding and `forward(x)` the five logits. The ECGFounder network keeps its original output layer with 150 labels, but it is not used.
The names of the modules must stay as they are, because the saved fine-tuned models are loaded by these names.

**What is trained.** In ECGFounder all layers are trained (in the teacher only the last two of the seven stages, see notebook 03).
In HuBERT-ECG and ECG-FM the convolutional feature extractor is fixed and all transformer layers are trained, as in the published fine-tuning of both models.
Masking of the input and layer drop are off. Dropout follows the published fine-tuning settings:
none in HuBERT-ECG, and 0.1 after the activation of the feed-forward layers in ECG-FM. ECGFounder is fine-tuned without dropout, as in its repository;
the option `dropout` exists only for the comparison with Monte Carlo dropout.

In [ ]:
import json
import sys
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"                      # mixed precision (float16 inside the networks) in training and in reading, for all three models
EMBED_DIM = {"ecgfounder": 1024, "hubert": 768, "ecgfm": 768}
FE_CHUNK = 128                                   # the fixed convolutional feature extractor of the transformers runs in chunks of this many inputs (its activations are large)


# ------------------------------------------------------------------------------------------------------------------ ECGFounder
def _ecgfounder_network(dropout=0.0):
    """The network of ECGFounder (class Net1D of the ECGFounder repository) with the settings of the released 12-lead model.
    The residual blocks contain dropout layers after each activation; the repository leaves them unused (use_do=False) and defines them with p = 0.5.
    With dropout > 0 they are switched on and set to the requested rate."""
    if str(ECGFOUNDER_CODE) not in sys.path:
        sys.path.insert(0, str(ECGFOUNDER_CODE))
    from net1d import Net1D                       # not part of this repository: clone the ECGFounder repository first (notebook 03)
    net = Net1D(in_channels=12, base_filters=64, ratio=1, filter_list=[64, 160, 160, 400, 400, 1024, 1024], m_blocks_list=[2, 2, 2, 3, 3, 4, 4],
                kernel_size=16, stride=2, groups_width=16, verbose=False, use_bn=False, use_do=dropout > 0, n_classes=150, return_features=True)
    for m in net.modules():
        if isinstance(m, nn.Dropout):
            m.p = dropout
    return net


def _ecgfounder_pretrained():
    """Weights of the released ECGFounder model. The file holds a few NumPy scalars besides the tensors; they are allowed by name,
    so that the file can be read with weights_only=True (no code from the file is executed)."""
    ma = np._core.multiarray
    allow = [(ma.scalar, "numpy.core.multiarray.scalar"), (ma.scalar, "numpy._core.multiarray.scalar"), (np.dtype, "numpy.dtype"),
             (np.dtypes.Float64DType, "numpy.dtypes.Float64DType"), (np.dtypes.Int64DType, "numpy.dtypes.Int64DType"), (np.dtypes.Float32DType, "numpy.dtypes.Float32DType")]
    with torch.serialization.safe_globals(allow):
        return torch.load(ECGFOUNDER_WEIGHTS, map_location="cpu", weights_only=True)["state_dict"]


def _standardize_all(x):
    """Mean 0 and SD 1 over all leads and time of each input (the input convention of ECGFounder)."""
    x = x.float()
    return (x - x.mean(dim=(-2, -1), keepdim=True)) / (x.std(dim=(-2, -1), keepdim=True, correction=0) + 1e-8)


# ------------------------------------------------------------------------------------------------------------------ HuBERT-ECG
def _rename_weight_norm(sd, target_keys, prefix="encoder.pos_conv_embed.conv."):
    """The two tensors of the weight-normalised positional convolution have different names in different versions of PyTorch
    (weight_g / weight_v or parametrizations.weight.original0 / original1). Rename them to the names that the model in memory uses."""
    new = prefix + "parametrizations.weight.original0" in target_keys
    pairs = {"weight_g": "parametrizations.weight.original0", "weight_v": "parametrizations.weight.original1"}
    if not new:
        pairs = {v: k for k, v in pairs.items()}
    return {(prefix + pairs[k[len(prefix):]] if k.startswith(prefix) and k[len(prefix):] in pairs else k): v for k, v in sd.items()}


class HubertECG(nn.Module):
    """HuBERT-ECG (base size) as a feature extractor: (B, 12, L) at 500 Hz -> (B, 768).
    The released weights are loaded into the standard HubertModel of the transformers library, from which the published model class is derived;
    the code of the HuBERT-ECG repository is not needed. The tensors that only pretraining uses (prediction head, mask token) are not loaded."""
    SLOT = 500                                    # samples of one lead in the input of the network: 5 s at 100 Hz

    def __init__(self, pretrained=True):
        super().__init__()
        from transformers import HubertConfig, HubertModel
        cfg = json.loads((HUBERT_DIR / "config.json").read_text())
        for k in ("auto_map", "architectures", "model_type", "ensemble_length", "vocab_sizes", "torch_dtype", "transformers_version"):
            cfg.pop(k, None)                      # entries of the published model class that HubertConfig does not know
        cfg.update(hidden_dropout=0.0, activation_dropout=0.0, attention_dropout=0.0, feat_proj_dropout=0.0, final_dropout=0.0, layerdrop=0.0,
                   mask_time_prob=0.0, mask_feature_prob=0.0, apply_spec_augment=False)
        self.m = HubertModel(HubertConfig(**cfg))
        self.load_report = None
        if pretrained:
            from safetensors.torch import load_file
            sd = _rename_weight_norm(load_file(str(HUBERT_DIR / "model.safetensors")), set(self.m.state_dict().keys()))
            res = self.m.load_state_dict(sd, strict=False)
            self.load_report = {"loaded": len(sd) - len(res.unexpected_keys), "missing": list(res.missing_keys), "not_used": list(res.unexpected_keys)}
        self.m.feature_extractor.requires_grad_(False)      # the convolutional feature extractor stays fixed
        # gradient checkpointing: the same gradients with much less memory and about a quarter more time; without it, distillation does not fit into 10 GB
        self.m.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        from scipy.signal import firwin
        # low-pass filter (63 taps, 45 Hz) applied to each lead before every fifth sample is taken (500 Hz -> 100 Hz)
        self.register_buffer("fir", torch.tensor(firwin(63, 45.0, fs=500.0), dtype=torch.float32).view(1, 1, -1), persistent=False)

    def prepare(self, x):
        """(B, 12, L) at 500 Hz -> (B, 6000): the input of the network, 12 leads x 5 s at 100 Hz, lead after lead."""
        B, C, L = x.shape
        x = x.float()
        lo, hi = x.amin(-1, keepdim=True), x.amax(-1, keepdim=True)
        x = 2 * (x - lo) / (hi - lo + 1e-8) - 1                    # each lead to the range -1 to 1 within the input
        k = self.fir.shape[-1] // 2
        x = F.conv1d(F.pad(x.reshape(B * C, 1, L), (k, k), mode="reflect"), self.fir, stride=5).view(B, C, -1)
        n = x.shape[-1]
        if n < self.SLOT:                                          # shorter than 5 s: repeat the input within the position of each lead
            x = torch.cat([x] * (self.SLOT // n + 1), -1)[..., :self.SLOT]
        return x.reshape(B, -1)

    def forward(self, x):
        with torch.autocast(device_type=x.device.type, enabled=False):
            inp = self.prepare(x)                                  # the preparation of the input is always in single precision
        with torch.no_grad():
            f = torch.cat([self.m.feature_extractor(c) for c in inp.split(FE_CHUNK)])
        h = self.m.feature_projection(f.transpose(1, 2))
        h = h[0] if isinstance(h, tuple) else h
        return self.m.encoder(h).last_hidden_state.mean(1)


# ------------------------------------------------------------------------------------------------------------------ ECG-FM
def _ecgfm_names(sd, target_keys):
    """Rename the tensors of the ECG-FM checkpoint (fairseq-signals) to the names of transformers.Wav2Vec2Model, which computes the same network
    (convolutional features -> layer norm -> projection -> + convolutional positional embedding -> layer norm -> 12 transformer layers).
    Returns the renamed tensors and the names that only pretraining uses (quantizer, projection heads)."""
    out, not_used = {}, []
    for k, v in sd.items():
        p = k.split(".")
        if k.startswith("feature_extractor.conv_layers."):
            nk = f"feature_extractor.conv_layers.{p[2]}." + ("conv." if p[3] == "0" else "layer_norm.") + p[4]
        elif k.startswith("layer_norm."):
            nk = "feature_projection." + k
        elif k.startswith("post_extract_proj."):
            nk = "feature_projection.projection." + p[1]
        elif k.startswith("conv_pos.pos_conv.0."):
            nk = "encoder.pos_conv_embed.conv." + p[3]
        elif k.startswith("encoder.layers."):
            part = {"self_attn": "attention." + ".".join(p[4:]), "self_attn_layer_norm": "layer_norm." + p[-1],
                    "fc1": "feed_forward.intermediate_dense." + p[-1], "fc2": "feed_forward.output_dense." + p[-1]}.get(p[3], ".".join(p[3:]))
            nk = f"encoder.layers.{p[2]}.{part}"
        elif k.startswith("encoder.layer_norm."):
            nk = k
        elif k == "mask_emb":
            nk = "masked_spec_embed"
        else:
            not_used.append(k); continue
        out[nk] = v
    return _rename_weight_norm(out, target_keys), not_used


class ECGFM(nn.Module):
    """ECG-FM as a feature extractor: (B, 12, L) at 500 Hz -> (B, 768).
    The network is a wav2vec 2.0 model whose first convolution takes the 12 leads as input channels (four convolutions, kernel 2, stride 2: one token per 32 ms).
    It is built from transformers.Wav2Vec2Model with the sizes stored in the checkpoint (12 layers, 768 dimensions), and the released weights are loaded under new names."""

    def __init__(self, pretrained=True):
        super().__init__()
        from transformers import Wav2Vec2Config, Wav2Vec2Model
        cfg = Wav2Vec2Config(hidden_size=768, num_hidden_layers=12, num_attention_heads=12, intermediate_size=3072, hidden_act="gelu",
                             hidden_dropout=0.0, activation_dropout=0.1, attention_dropout=0.0, feat_proj_dropout=0.0, final_dropout=0.0, layerdrop=0.0,
                             feat_extract_norm="group", feat_extract_activation="gelu", conv_dim=(256, 256, 256, 256), conv_stride=(2, 2, 2, 2), conv_kernel=(2, 2, 2, 2),
                             conv_bias=False, num_conv_pos_embeddings=128, num_conv_pos_embedding_groups=16, do_stable_layer_norm=False,
                             apply_spec_augment=False, mask_time_prob=0.0, mask_feature_prob=0.0, layer_norm_eps=1e-5)
        self.m = Wav2Vec2Model(cfg)
        self.m.feature_extractor.conv_layers[0].conv = nn.Conv1d(12, 256, 2, stride=2, bias=False)      # 12 leads as input channels
        self.load_report = None
        if pretrained:
            ck = torch.load(ECGFM_WEIGHTS, map_location="cpu", weights_only=True)
            sd, not_used = _ecgfm_names(ck["model"], set(self.m.state_dict().keys()))
            res = self.m.load_state_dict(sd, strict=False)
            self.load_report = {"loaded": len(sd) - len(res.unexpected_keys), "missing": list(res.missing_keys), "not_used": not_used + list(res.unexpected_keys)}
        self.m.feature_extractor.requires_grad_(False)      # the convolutional feature extractor stays fixed
        self.m.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})    # see HubertECG

    def prepare(self, x):
        """Each lead to mean 0 and SD 1 within the input (a constant lead becomes 0)."""
        x = x.float()
        x = x - x.mean(-1, keepdim=True)
        return x / x.pow(2).mean(-1, keepdim=True).sqrt().clamp_min(1e-6)

    def forward(self, x):
        with torch.autocast(device_type=x.device.type, enabled=False):
            h = self.prepare(x)
        with torch.no_grad():
            parts = []
            for c in h.split(FE_CHUNK):
                for layer in self.m.feature_extractor.conv_layers:
                    c = layer(c)
                parts.append(c)
            h = torch.cat(parts)
        h = self.m.feature_projection(h.transpose(1, 2))
        h = h[0] if isinstance(h, tuple) else h
        return self.m.encoder(h).last_hidden_state.mean(1)


# ------------------------------------------------------------------------------------------------------------------ one class for the three models
class FoundationModel(nn.Module):
    """A foundation model with a linear layer of five outputs on its embedding.

    name        "ecgfounder" | "hubert" | "ecgfm"
    dropout     ECGFounder only: rate of the dropout in the residual blocks (0 = none, as in all models except those for Monte Carlo dropout)
    partial     ECGFounder only: train the last two of the seven stages and the head, and keep the rest fixed (used for the teacher)
    pretrained  load the released weights (False when a fine-tuned model is loaded afterwards)
    Input of embed() and forward(): float tensor (B, 12, L), 500 Hz, mV, cut from the cached recordings."""

    def __init__(self, name, dropout=0.0, partial=False, pretrained=True):
        super().__init__()
        assert name in MODELS, name
        assert name == "ecgfounder" or (dropout == 0 and not partial), "dropout and partial fine-tuning are defined for ECGFounder only"
        self.name, self.dim = name, EMBED_DIM[name]
        if name == "ecgfounder":
            self.bb = _ecgfounder_network(dropout)
            self.load_report = None
            if pretrained:
                res = self.bb.load_state_dict(_ecgfounder_pretrained(), strict=False)
                self.load_report = {"loaded": len(self.bb.state_dict()) - len(res.missing_keys), "missing": list(res.missing_keys), "not_used": list(res.unexpected_keys)}
            if partial:
                self.bb.requires_grad_(False)
                self.bb.stage_list[5:].requires_grad_(True)
        else:
            self.bb = HubertECG(pretrained) if name == "hubert" else ECGFM(pretrained)
            self.load_report = self.bb.load_report
        self.head = nn.Linear(self.dim, len(CLASSES))

    def embed(self, x):
        if self.name == "ecgfounder":
            with torch.autocast(device_type=x.device.type, enabled=False):
                x = _standardize_all(x)
            return self.bb(x)[1]
        return self.bb(x)

    def forward(self, x):
        return self.head(self.embed(x))


def parse_tag(tag):
    """"ecgfounder_dropout30_s1" -> ("ecgfounder", "dropout30", 1)"""
    model, kind, seed = tag.split("_")
    return model, kind, int(seed[1:])


def kind_settings(model, kind):
    """What distinguishes the kinds of fine-tuned model (see the list of kinds in the constants above)."""
    assert kind in ("baseline", "teacher", "student", "windows", "agreement", "selfdistill") or kind.startswith("dropout"), kind
    return {"window_input": kind in ("teacher", "windows"),                         # trained on random 2.5-s windows instead of whole segments
            "partial": model == "ecgfounder" and kind == "teacher",                # the ECGFounder teacher was trained with the last two stages only
            "dropout": int(kind[7:]) / 100 if kind.startswith("dropout") else 0.0,
            "target_from": {"student": "teacher", "selfdistill": "baseline"}.get(kind),    # kind of the model of the same seed whose window average is the distillation target
            "agreement": kind == "agreement"}


def load_finetuned(tag, model_dir=None, device=None):
    """Load a fine-tuned model by its tag from FINETUNED (or model_dir), in evaluation mode. Every tensor of the file must match the model."""
    model, kind, _ = parse_tag(tag)
    net = FoundationModel(model, dropout=kind_settings(model, kind)["dropout"], pretrained=False)
    ck = torch.load((model_dir or FINETUNED) / f"{tag}.pt", map_location="cpu", weights_only=True)
    net.load_state_dict(ck["state_dict"])
    return net.to(device or DEVICE).eval()

## Cached recordings and window layouts

The filtered recordings are stored as arrays of shape `(n, 12, 5000)` in float16 (notebook 01). They are memory-mapped, so that only the rows that are needed are read from disk.
The tables of the sets give the array and the row of every recording (notebook 02).

All window layouts are defined within a 5-s segment:

- **window reading**: 25 windows of 2.5 s with evenly spaced onsets, the first at the start of the segment and the last ending at its end (onsets about 0.1 s apart);
- **teacher target**: 7 windows of 2.5 s, placed in the same way (onsets about 0.42 s apart);
- **views of the penalized head**: two views of 4.5 s that start 0.5 s apart (at 0 s and at 0.5 s of the segment).

In [ ]:
_ARRAYS = {}


def open_array(name):
    """Memory-mapped array of cached recordings: "ptbxl", "mimic_pool", "mimic_eval", "echonext" or "code15/part{i}" (the names used in column `array` of EVAL_RECORDS)."""
    if name not in _ARRAYS:
        fixed = {"ptbxl": PTBXL_ARRAY, "mimic_pool": MIMIC_POOL_ARRAY, "mimic_eval": MIMIC_EVAL_ARRAY, "echonext": ECHONEXT_ARRAY}
        path = fixed[name] if name in fixed else CODE15_CACHE / f"{name.split('/')[1]}_filtered_f16.npy"
        _ARRAYS[name] = np.load(path, mmap_mode="r")
    return _ARRAYS[name]


def read_crops(array, rows, starts=0, length=N10):
    """Pieces of recordings as float32 (len(rows), 12, length): row rows[i] of the array from sample starts[i] (one number for all rows, or one per row)."""
    starts = np.broadcast_to(np.asarray(starts, dtype=int), (len(rows),))
    return np.stack([np.asarray(array[r, :, s:s + length]) for r, s in zip(rows, starts)]).astype(np.float32)


def read_records(table):
    """The 10-s recordings of the rows of a table with the columns `array` and `row` (for example rows of EVAL_RECORDS), float32 (n, 12, 5000)."""
    return np.stack([np.asarray(open_array(a)[r]) for a, r in zip(table["array"].values, table["row"].values)]).astype(np.float32)


def window_starts(n_windows, length=WIN, total=SEG):
    """Onsets (in samples) of n_windows windows of `length` samples, evenly spaced from the start of a piece of `total` samples to its end."""
    return np.linspace(0, total - length, n_windows).round().astype(int)


READ_STARTS = window_starts(N_WIN_READ)           # 25 windows of a window reading
TEACHER_STARTS = window_starts(N_WIN_TEACHER)     # 7 windows of the teacher target
VIEW_STARTS = np.array([0, VIEW_SHIFT])           # the two 4.5-s views of the penalized head


def cut_pieces(x, starts, length):
    """(B, 12, L) -> (B, len(starts), 12, length): the pieces of every row that begin at `starts` (tensor in, tensor out)."""
    return torch.stack([x[..., s:s + length] for s in starts], 1)


def two_segments(x):
    """(B, 12, 5000) -> (B, 2, 12, 2500): the two adjacent 5-s segments of 10-s recordings."""
    return cut_pieces(x, (0, SEG), SEG)


@torch.no_grad()
def embed_inputs(net, x, batch_size=512):
    """Embeddings of inputs x (n, 12, L) (array or tensor) as a float32 array (n, D), computed in batches with mixed precision."""
    out = []
    for t in torch.as_tensor(x).split(batch_size):
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=AMP):
            out.append(net.embed(t.to(DEVICE, torch.float32)).float())
    return torch.cat(out).cpu().numpy()


@torch.no_grad()
def model_logits(net, x, batch_size=512):
    """Logits (n, 5) of the model head for inputs x (n, 12, L) (array or tensor), as a float32 tensor on DEVICE, computed in batches with mixed precision."""
    out = []
    for t in torch.as_tensor(x).split(batch_size):
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=AMP):
            out.append(net(t.to(DEVICE, torch.float32)).float())
    return torch.cat(out)


def training_tables(subset=None):
    """Rows and labels of the PTB-XL training and validation recordings and rows of the unlabelled pool.
    subset = (n_train, n_val, n_pool) keeps only the first recordings of each set (functional tests)."""
    T = pd.read_csv(PTBXL_SETS); tr, va = T[T.set == "train"], T[T.set == "val"]
    out = {"train_rows": tr.row.values, "train_labels": tr[CLASSES].values.astype(np.float32), "val_rows": va.row.values, "val_labels": va[CLASSES].values.astype(np.float32),
           "pool_rows": pd.read_csv(POOL_RECORDS).row.values}
    if subset is not None:
        n_train, n_val, n_pool = subset
        out = {k: v[:{"train": n_train, "val": n_val, "pool": n_pool}[k.split("_")[0]]] for k, v in out.items()}
    return out

## Fine-tuning

One function, `finetune`, trains every kind of model of the study. It is defined here, and not in notebook 03, because the cost notebook (06) times the same function.

**Common frame** (all models, all kinds)

- An example is one 5-s segment: at every step 128 PTB-XL training recordings are drawn with replacement, and one of the two segments of each is taken at random.
- Loss for the labels: binary cross-entropy on the five outputs, with the positive class of each diagnosis weighted by (1 − prevalence) / prevalence in the training set.
- AdamW with weight decay 0.01, one-cycle schedule of the learning rate with 10% warm-up, 3,000 steps, mixed precision, no gradient clipping.
  The learning rate of the head is 1e-3; that of the network is 1e-4 in ECGFounder and 1e-5 in HuBERT-ECG and ECG-FM.
- Every 250 steps and after the last step, the macro AUROC of the five classes is calculated in the validation fold, separately for the first and for the second segment, and averaged.
  The weights of the step with the highest value are saved. (For models trained on 2.5-s windows the validation inputs are four fixed windows of each 10-s recording, at 0, 2.5, 5 and 7.5 s.)

**Kinds**

| Kind | Input in training | Added to the loss |
|---|---|---|
| `baseline` | whole segment | – |
| `teacher`, `windows` | a random 2.5-s window of the segment | – |
| `student`, `selfdistill` | whole segment | distillation (below) |
| `agreement` | whole segment | agreement loss (below) |
| `dropoutNN` | whole segment | – (dropout in the residual blocks at rate 0.NN) |

*Distillation.* The target of a segment is the mean logit of another model over seven 2.5-s windows of that segment; it is calculated once, before training (`compute_teacher_targets`).
Each step adds a batch of 128 unlabelled segments from the MIMIC-IV-ECG pool, and the loss is
cross-entropy(labels) + 0.5 × [MSE(logits, target) in the labelled batch + MSE(logits, target) in the unlabelled batch], where MSE is the mean over examples and classes.
The student reads the whole segment once, so it learns to give in one read what the other model gives on average over the windows.
For `student` the other model is the teacher of the same seed, and for `selfdistill` the baseline model of the same seed.

*Agreement loss.* Each step adds 64 PTB-XL training segments and 64 pool segments, each as two 4.5-s views whose onsets are drawn independently between 0 and 0.5 s.
For each class, the loss is the variance of the logit within the pairs of views divided by the sum of the variances within and between pairs,
averaged over the classes and over the two sources, and added to the cross-entropy with weight 1.

**Random numbers.** The seed sets the initial weights of the head, the order of the labelled batches (one generator) and that of the unlabelled batches (a second generator).
Models of the same foundation model and seed therefore start from the same head, and a baseline and a student of the same seed see the same labelled segments in the same order.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

TRAIN_STEPS = 3000                                # optimizer steps of every fine-tuning run
EVAL_EVERY = 250                                  # the validation AUROC is calculated every 250 steps and after the last step
BATCH = 128                                       # labelled segments per step, and unlabelled segments per step in distillation
BATCH_VIEWS = 64                                  # segments per source and step in the agreement loss
LR_NETWORK = {"ecgfounder": 1e-4, "hubert": 1e-5, "ecgfm": 1e-5}
LR_HEAD = 1e-3
WEIGHT_DECAY = 1e-2
WARMUP = 0.1                                      # fraction of the steps in which the learning rate rises (one-cycle schedule)
QUICK_SUBSET = (512, 256, 256)                    # training, validation and pool recordings used when a notebook runs with QUICK_TEST


def agreement_loss(za, zb):
    """Disagreement of the logits of two views (za, zb: (n, classes)): for each class the variance within pairs / (variance within + variance between pairs), then the mean over classes."""
    within = ((za - zb) ** 2).mean(0) / 2
    between = ((za + zb) / 2).var(0)
    return (within / (within + between + 1e-6)).mean()


@torch.no_grad()
def validation_auroc(net, rows, labels, window_input=False):
    """Macro AUROC of the five classes in the validation recordings, calculated for each reading position and averaged.
    Positions: the two segments, or, for models trained on 2.5-s windows, four fixed windows of the 10-s recording."""
    was_training = net.training; net.eval()
    starts, length = (window_starts(4, WIN, N10), WIN) if window_input else ((0, SEG), SEG)
    array = open_array("ptbxl"); p = np.zeros((len(starts), len(rows), len(CLASSES)), np.float32)
    for a in range(0, len(rows), 64):
        x = torch.from_numpy(read_crops(array, rows[a:a + 64])).to(DEVICE)
        z = model_logits(net, torch.cat([x[..., s:s + length] for s in starts]), 256)
        p[:, a:a + len(x)] = torch.sigmoid(z).cpu().numpy().reshape(len(starts), len(x), -1)
    net.train(was_training)
    return float(np.mean([[roc_auc_score(labels[:, j], p[k, :, j]) for j in range(len(CLASSES))] for k in range(len(starts))]))


def finetune(model, kind, seed, steps=TRAIN_STEPS, eval_every=EVAL_EVERY, subset=None, model_dir=None, log_dir=None, target_dir=None, verbose=True):
    """Fine-tune one foundation model and save the weights of the step with the highest validation AUROC.

    model, kind, seed   which model to train (see kind_settings); the file is model_dir / "{model}_{kind}_s{seed}.pt"
    steps, eval_every   3,000 and 250 in the study
    subset              (n_train, n_val, n_pool) for a functional test with the first recordings of each set
    model_dir, log_dir, target_dir   default FINETUNED, TRAINING_LOGS, TEACHER_TARGETS
    Returns a dictionary with the tag, the selected step, its validation AUROC, the time in minutes and the peak GPU memory in GB."""
    model_dir, log_dir, target_dir = model_dir or FINETUNED, log_dir or TRAINING_LOGS, target_dir or TEACHER_TARGETS
    for d in (model_dir, log_dir):
        d.mkdir(parents=True, exist_ok=True)
    tag, cfg = model_tag(model, kind, seed), kind_settings(model, kind)
    t0 = time.time(); torch.manual_seed(seed); np.random.seed(seed)
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    D = training_tables(subset); tr_rows, Y, pool_rows = D["train_rows"], D["train_labels"], D["pool_rows"]
    P, M = open_array("ptbxl"), open_array("mimic_pool")
    if cfg["target_from"]:                        # distillation targets, calculated beforehand by compute_teacher_targets()
        z = np.load(target_dir / f"{model_tag(model, cfg['target_from'], seed)}.npz"); T_lab, T_unl = z["ptbxl"], z["pool"]
        assert len(T_lab) == len(tr_rows) and len(T_unl) == len(pool_rows), "the targets were calculated for other sets of recordings"

    net = FoundationModel(model, dropout=cfg["dropout"], partial=cfg["partial"]).to(DEVICE)
    network = [p for n, p in net.named_parameters() if p.requires_grad and not n.startswith("head")]
    opt = torch.optim.AdamW([{"params": network, "lr": LR_NETWORK[model]}, {"params": net.head.parameters(), "lr": LR_HEAD}], weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[LR_NETWORK[model], LR_HEAD], total_steps=steps, pct_start=WARMUP)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=AMP)
    prevalence = Y.mean(0); pos_weight = torch.tensor((1 - prevalence) / np.maximum(prevalence, 1e-3), device=DEVICE)
    rng_lab, rng_unl = np.random.default_rng(seed), np.random.default_rng(1000 + seed)      # labelled batches; unlabelled batches and views
    if verbose:
        print(f"[{tag}] trainable parameters {sum(p.numel() for p in net.parameters() if p.requires_grad) / 1e6:.1f} M | training recordings {len(tr_rows):,} | "
              f"pool recordings {len(pool_rows):,} | steps {steps}", flush=True)

    # With mixed precision, the gradient scaler skips an optimizer step when the scaled gradients overflow and then lowers its scale (usual in the first steps).
    # PyTorch then warns that the schedule advanced without the optimizer; this is expected here and the warning is switched off.
    warnings.filterwarnings("ignore", message="Detected call of `lr_scheduler.step")
    best_auroc, best_step, best_state, log = -1.0, -1, None, []
    loss_extra = torch.zeros((), device=DEVICE); net.train()
    for step in range(steps):
        i = rng_lab.choice(len(tr_rows), BATCH); h = rng_lab.integers(0, 2, BATCH)           # recordings (with replacement) and which of their two segments
        if cfg["window_input"]:
            start, length = h * SEG + rng_lab.integers(0, SEG - WIN + 1, BATCH), WIN         # a random 2.5-s window inside the segment
        else:
            start, length = h * SEG, SEG
        x = torch.from_numpy(read_crops(P, tr_rows[i], start, length)).to(DEVICE); y = torch.from_numpy(Y[i]).to(DEVICE)
        if cfg["target_from"]:
            j = rng_unl.choice(len(pool_rows), BATCH); g = rng_unl.integers(0, 2, BATCH)
            xu = torch.from_numpy(read_crops(M, pool_rows[j], g * SEG, SEG)).to(DEVICE)
        if cfg["agreement"]:
            views = []
            for array, rows in ((P, tr_rows), (M, pool_rows)):
                r = rows[rng_unl.choice(len(rows), BATCH_VIEWS)]; s = rng_unl.integers(0, 2, BATCH_VIEWS) * SEG
                views += [read_crops(array, r, s + rng_unl.integers(0, VIEW_SHIFT + 1, BATCH_VIEWS), VIEW) for _ in range(2)]
            xv = torch.from_numpy(np.concatenate(views)).to(DEVICE)
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=AMP):
            z = net(x).float()
            loss_labels = F.binary_cross_entropy_with_logits(z, y, pos_weight=pos_weight)
            if cfg["target_from"]:
                zu = net(xu).float()
                loss_extra = 0.5 * (F.mse_loss(z, torch.from_numpy(T_lab[i, h]).to(DEVICE)) + F.mse_loss(zu, torch.from_numpy(T_unl[j, g]).to(DEVICE)))
            if cfg["agreement"]:
                zv = net(xv).float().view(4, BATCH_VIEWS, -1)                               # PTB-XL view 1, view 2, pool view 1, view 2
                loss_extra = 0.5 * (agreement_loss(zv[0], zv[1]) + agreement_loss(zv[2], zv[3]))
            loss = loss_labels + loss_extra
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        if step % eval_every == 0 or step == steps - 1:
            auroc = float("nan")
            if step >= eval_every:
                auroc = validation_auroc(net, D["val_rows"], D["val_labels"], cfg["window_input"])
                if auroc > best_auroc:
                    best_auroc, best_step, best_state = auroc, step, {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
            log.append({"step": step, "loss_labels": float(loss_labels), "loss_extra": float(loss_extra), "val_auroc": auroc, "minutes": (time.time() - t0) / 60})
            pd.DataFrame(log).to_csv(log_dir / f"{tag}.csv", index=False)
            if verbose:
                print(f"[{tag}] step {step:4d} | loss: labels {float(loss_labels):.4f}, extra {float(loss_extra):.4f} | validation AUROC {auroc:.4f} | {(time.time() - t0) / 60:.1f} min", flush=True)

    assert best_state is not None and all(torch.isfinite(v).all() for v in best_state.values() if v.is_floating_point())
    tmp = model_dir / f"{tag}.pt.tmp"                 # written under another name first, so that an interrupted run leaves no incomplete model file
    torch.save({"state_dict": best_state, "tag": tag, "model": model, "kind": kind, "seed": seed, "selected_step": best_step, "val_auroc": best_auroc, "steps": steps, "classes": CLASSES}, tmp)
    tmp.replace(model_dir / f"{tag}.pt")
    info = {"tag": tag, "selected_step": best_step, "val_auroc": best_auroc, "minutes": (time.time() - t0) / 60,
            "peak_gpu_gb": torch.cuda.max_memory_allocated() / 2 ** 30 if DEVICE.type == "cuda" else float("nan")}
    if verbose:
        print(f"[{tag}] selected step {best_step} (validation AUROC {best_auroc:.4f}) | peak GPU memory {info['peak_gpu_gb']:.2f} GB | {info['minutes']:.1f} min", flush=True)
    del net, opt, scaler
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return info


@torch.no_grad()
def window_average_logits(net, array, rows, starts=TEACHER_STARTS, batch_records=64):
    """Mean logit of the model head over the 2.5-s windows that begin at `starts`, for both segments of the recordings: float32 (len(rows), 2, 5)."""
    out = np.zeros((len(rows), 2, len(CLASSES)), np.float32)
    for a in range(0, len(rows), batch_records):
        x = two_segments(torch.from_numpy(read_crops(array, rows[a:a + batch_records])).to(DEVICE))        # (B, 2, 12, 2500)
        for h in (0, 1):
            w = cut_pieces(x[:, h], starts, WIN)                                                                  # (B, windows, 12, 1250)
            out[a:a + len(x), h] = model_logits(net, w.flatten(0, 1)).view(len(x), len(starts), -1).mean(1).cpu().numpy()
    return out


def compute_teacher_targets(teacher_tag, subset=None, model_dir=None, target_dir=None, verbose=True):
    """Distillation targets of one model: its mean logit over seven 2.5-s windows of every PTB-XL training segment and of every segment of the unlabelled pool.
    Writes target_dir / "{teacher_tag}.npz" with `ptbxl` (training recordings, 2, 5) and `pool` (pool recordings, 2, 5), rows in the order of the tables of the sets."""
    target_dir = target_dir or TEACHER_TARGETS; target_dir.mkdir(parents=True, exist_ok=True)
    t0 = time.time(); D = training_tables(subset); net = load_finetuned(teacher_tag, model_dir)
    ptbxl = window_average_logits(net, open_array("ptbxl"), D["train_rows"]); pool = window_average_logits(net, open_array("mimic_pool"), D["pool_rows"])
    assert np.isfinite(ptbxl).all() and np.isfinite(pool).all()
    tmp = target_dir / f"{teacher_tag}.tmp.npz"; np.savez(tmp, ptbxl=ptbxl, pool=pool); tmp.replace(target_dir / f"{teacher_tag}.npz")
    del net
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    if verbose:
        print(f"[{teacher_tag}] targets for {len(ptbxl):,} training and {len(pool):,} pool recordings | {(time.time() - t0) / 60:.1f} min", flush=True)
    return {"tag": teacher_tag, "minutes": (time.time() - t0) / 60}

## Penalized head

With the network fixed, the model head is replaced by a linear head that is refitted on the embeddings of the PTB-XL training segments:

1. **Embeddings.** For both segments of every training recording: the embedding of the whole segment, and the embeddings of two 4.5-s views of the segment that start 0.5 s apart.
2. **Standardization.** Every dimension of the embedding is centred and scaled with the mean and SD of the embeddings of the training segments.
3. **Covariance of the differences.** S is the covariance of (view 1 − view 2) / √2 of the standardized embeddings, scaled so that its trace equals the number of dimensions.
   S describes the directions in which the embedding moves when only the position of the view within the same segment changes. No label and no second recording is needed for it.
4. **Fit.** For each diagnostic class, a logistic regression (L2 penalty with C = 0.01, balanced class weights) is fitted to the standardized embeddings after they are multiplied by (I + γS)^(−1/2).
   If v are the fitted weights, the weights for the standardized embedding are w = (I + γS)^(−1/2) v, and the ridge penalty on v equals wᵀw + γ wᵀSw:
   the fit is a ridge logistic regression with the added penalty γ wᵀSw, which discourages weights along the directions that move between views.
   With γ = 0 the multiplication changes nothing, and the head is refitted without a penalty.

The logit of a new input is ((embedding − mean) / SD) · w + intercept. The fitted heads are saved, so that a model can be used later without the training data.

In [ ]:
PENALIZED_HEADS = RESULTS / "penalized_heads"     # {tag}.npz: the refitted heads of a fine-tuned model (written by notebook 04)
PENALIZED_HEADS.mkdir(parents=True, exist_ok=True)


def fit_penalized_heads(net, gammas=PENALTY_WEIGHTS, n_train=None, verbose=True):
    """Fit the penalized heads of a fine-tuned model on the PTB-XL training segments, one head for each penalty weight in `gammas`.

    Returns a dictionary: gammas (G,), mean and sd (D,) of the standardization, W (G, D, 5) and b (G, 5) for the standardized embedding,
    and the time in seconds of the two parts (embeddings on the GPU, fits on the CPU).
    n_train keeps only the first training recordings (functional tests)."""
    D = training_tables(None if n_train is None else (n_train, 1, 1)); rows, Y = D["train_rows"], D["train_labels"]
    array, dim, n = open_array("ptbxl"), net.dim, len(rows)
    t0 = time.time(); E_seg = np.zeros((n, 2, dim), np.float32); E_view = np.zeros((n, 2, 2, dim), np.float32)
    for a in range(0, n, 128):
        x = two_segments(torch.from_numpy(read_crops(array, rows[a:a + 128])).to(DEVICE))                  # (B, 2, 12, 2500)
        E_seg[a:a + len(x)] = embed_inputs(net, x.flatten(0, 1)).reshape(len(x), 2, dim)
        views = cut_pieces(x.flatten(0, 1), VIEW_STARTS, VIEW)                                                    # (B * 2, 2, 12, 2250)
        E_view[a:a + len(x)] = embed_inputs(net, views.flatten(0, 1)).reshape(len(x), 2, 2, dim)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    t1 = time.time()
    X = E_seg.reshape(-1, dim); Yx = np.repeat(Y, 2, 0)                                                    # one example per segment
    mean, sd = X.mean(0), X.std(0) + 1e-6; X = (X - mean) / sd
    S = np.cov((((E_view[:, :, 0] - E_view[:, :, 1]).reshape(-1, dim) / sd) / np.sqrt(2)).T)               # covariance of the differences between the two views
    S *= dim / np.trace(S)
    e, U = np.linalg.eigh(S); e = np.maximum(e, 0)
    W = np.zeros((len(gammas), dim, len(CLASSES)), np.float32); b = np.zeros((len(gammas), len(CLASSES)), np.float32)
    for g, gamma in enumerate(gammas):
        A = (U / np.sqrt(1 + gamma * e)) @ U.T                                                              # (I + gamma S)^(-1/2)
        XA = X @ A
        for j in range(len(CLASSES)):
            fit = LogisticRegression(C=HEAD_C, max_iter=3000, class_weight="balanced").fit(XA, Yx[:, j])
            W[g, :, j] = A @ fit.coef_[0]; b[g, j] = fit.intercept_[0]
        if verbose:
            print(f"    penalty weight {gamma}: fitted | {(time.time() - t0) / 60:.1f} min", flush=True)
    return {"gammas": np.asarray(gammas), "mean": mean.astype(np.float32), "sd": sd.astype(np.float32), "W": W, "b": b,
            "seconds_embeddings": t1 - t0, "seconds_fit": time.time() - t1}


def save_heads(heads, tag, head_dir=None):
    head_dir = head_dir or PENALIZED_HEADS; head_dir.mkdir(parents=True, exist_ok=True)
    tmp = head_dir / f"{tag}.tmp.npz"; np.savez(tmp, **{k: heads[k] for k in ("gammas", "mean", "sd", "W", "b")}); tmp.replace(head_dir / f"{tag}.npz")


def load_heads(tag, head_dir=None):
    z = np.load((head_dir or PENALIZED_HEADS) / f"{tag}.npz")
    return {k: z[k] for k in z.files}


def head_logits(E, heads, gamma):
    """Logits (n, 5) of the penalized head with penalty weight gamma for embeddings E (n, D)."""
    g = list(heads["gammas"]).index(gamma)
    return ((E - heads["mean"]) / heads["sd"]) @ heads["W"][g] + heads["b"][g]

## Reading a segment

`read_probabilities` gives the probabilities that the decisions are based on, for the two ends of the comparison in the paper:
the baseline reading (one read of the segment with the model head) and the final setting (penalized head and mixed reading:
0.1 × the probability from one read of the segment + 0.9 × the mean probability over the 25 windows of 2.5 s, 26 forward passes for each segment).
Notebook 04 stores the logits of all readings instead, so that notebook 05 can form every combination of head and reading from them; this function is used to time the readings (notebook 06).

In [ ]:
@torch.no_grad()
def read_probabilities(net, x, heads=None, gamma=PENALTY_WEIGHT, mixed=False, batch_size=512):
    """Probabilities (n, 5), as a tensor on DEVICE, for 5-s segments x (n, 12, 2500) given as a tensor on DEVICE.

    heads=None    the model head; otherwise the penalized head with weight gamma from fit_penalized_heads() or load_heads()
    mixed=False   one read of each segment; True: mixed reading (MIX_SINGLE x single read + (1 - MIX_SINGLE) x mean over the 25 windows)"""
    if heads is None:
        weight, bias = net.head.weight.float().T, net.head.bias.float()
    else:
        g = list(heads["gammas"]).index(gamma); sd = torch.as_tensor(heads["sd"], device=DEVICE)
        weight = torch.as_tensor(heads["W"][g], device=DEVICE) / sd[:, None]
        bias = torch.as_tensor(heads["b"][g], device=DEVICE) - torch.as_tensor(heads["mean"], device=DEVICE) @ weight

    def prob(t):                                  # probabilities of inputs t (m, 12, L)
        out = []
        for c in t.split(batch_size):
            with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=AMP):
                out.append(net.embed(c).float())
        return torch.sigmoid(torch.cat(out) @ weight + bias)

    out = []
    for c in x.split(batch_size):                 # the windows of batch_size segments are cut at a time, to limit the memory
        p = prob(c)
        if mixed:
            pw = prob(cut_pieces(c, READ_STARTS, WIN).flatten(0, 1)).view(len(c), N_WIN_READ, -1).mean(1)
            p = MIX_SINGLE * p + (1 - MIX_SINGLE) * pw
        out.append(p)
    return torch.cat(out)